<a href="https://colab.research.google.com/github/AsimaZaheer/Stress-Detection/blob/main/integrate%20Transformer%2032HZ%20Murtaza%20V2%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# WESAD STRESS DETECTION
# 32 Hz + 3-Layer 1D CNN + Transformer + HRV
# LOSO (Leave-One-Subject-Out) Validation
# ============================================================

# IMPORTANT:
# Run this entire cell as ONE Python cell.
# Do NOT copy the ``` lines into the notebook.
# ============================================================

import os
import pickle
import zipfile
import subprocess
import sys

import numpy as np
from scipy import signal
from scipy.signal import find_peaks

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# ============================================================
# 1. CONFIGURATION
# ============================================================

SUBJECTS = [
    "S2", "S3", "S4", "S5", "S6", "S7",
    "S8", "S9", "S10", "S11", "S13", "S14",
    "S15", "S16", "S17"
]

TARGET_SAMPLING_RATE = 32
LABEL_NATIVE_RATE = 700

WINDOW_SIZE = 60       # seconds
STEP_SIZE = 10         # seconds

BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 1e-3

# Transformer settings.
# These are added only after the existing CNN feature extraction.
TRANSFORMER_HEADS = 4
TRANSFORMER_LAYERS = 2
TRANSFORMER_DROPOUT = 0.2

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ============================================================
# GOOGLE DRIVE DATASET STORAGE
# ============================================================

# WESAD ZIP download link
WESAD_ZIP_URL = (
    "https://drive.google.com/file/d/"
    "1MbfU2z4OnyevB0oX_yvHVue7Wv24KP7l"
    "/view?usp=sharing"
)

# Permanent Google Drive location
DRIVE_WESAD_DIR = "/content/drive/MyDrive/WESAD"

# ZIP will be stored permanently here
ZIP_PATH = os.path.join(
    DRIVE_WESAD_DIR,
    "WESAD.zip"
)

# Extracted WESAD dataset will also stay here
EXTRACT_PATH = DRIVE_WESAD_DIR


# ============================================================
# 2. MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

print("=" * 70)
print("MOUNTING GOOGLE DRIVE")
print("=" * 70)

drive.mount("/content/drive")

# Create WESAD folder if it does not exist
os.makedirs(
    DRIVE_WESAD_DIR,
    exist_ok=True
)

print("\nGoogle Drive mounted.")
print("WESAD storage location:")
print(DRIVE_WESAD_DIR)


# ============================================================
# 3. DOWNLOAD / EXTRACT DATASET
# ============================================================

def ensure_gdown():
    """Install gdown if it is not already installed."""

    try:
        import gdown
        return gdown

    except ImportError:

        print("Installing gdown...")

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "gdown"
            ]
        )

        import gdown

        return gdown


def find_subject_file(dataset_root, subject):
    """Recursively find Sx.pkl inside the extracted WESAD folder."""

    target_name = f"{subject}.pkl"

    direct_path = os.path.join(
        dataset_root,
        subject,
        target_name
    )

    if os.path.isfile(direct_path):
        return direct_path

    for root, _, files in os.walk(dataset_root):

        if target_name in files:

            return os.path.join(
                root,
                target_name
            )

    return None


def download_wesad_dataset():
    """
    Download and extract WESAD only if it is not
    already available in Google Drive.
    """

    print("=" * 70)
    print("WESAD DATASET SETUP")
    print("=" * 70)

    os.makedirs(
        DRIVE_WESAD_DIR,
        exist_ok=True
    )

    # --------------------------------------------------------
    # FIRST: Check whether WESAD is already extracted
    # --------------------------------------------------------

    existing_s2 = find_subject_file(
        EXTRACT_PATH,
        "S2"
    )

    if existing_s2 is not None:

        print("\nWESAD dataset already exists in Google Drive.")

        print("No download required.")
        print("No extraction required.")

        dataset_root = os.path.dirname(
            os.path.dirname(existing_s2)
        )

        print("\nS2.pkl found:")
        print(existing_s2)

        print("\nDataset root:")
        print(dataset_root)

        return dataset_root


    # --------------------------------------------------------
    # Dataset is not extracted.
    # Check whether ZIP already exists.
    # --------------------------------------------------------

    gdown = ensure_gdown()

    if not os.path.isfile(ZIP_PATH):

        print("\nWESAD ZIP not found in Google Drive.")

        print("Downloading WESAD ZIP...")

        gdown.download(
            WESAD_ZIP_URL,
            ZIP_PATH,
            quiet=False,
            fuzzy=True
        )

    else:

        print("\nWESAD ZIP already exists in Google Drive.")

        print(ZIP_PATH)


    # --------------------------------------------------------
    # Check ZIP
    # --------------------------------------------------------

    if not os.path.isfile(ZIP_PATH):

        raise FileNotFoundError(
            f"Download failed. ZIP was not created:\n"
            f"{ZIP_PATH}"
        )


    if not zipfile.is_zipfile(ZIP_PATH):

        raise RuntimeError(
            "Downloaded file is not a valid ZIP file. "
            "Check your Google Drive link and file permissions."
        )


    # --------------------------------------------------------
    # Extract ZIP only once
    # --------------------------------------------------------

    print("\nExtracting WESAD ZIP to Google Drive...")

    with zipfile.ZipFile(
        ZIP_PATH,
        "r"
    ) as zf:

        zf.extractall(
            EXTRACT_PATH
        )

    print("Extraction completed.")


    # --------------------------------------------------------
    # Find S2 after extraction
    # --------------------------------------------------------

    s2_file = find_subject_file(
        EXTRACT_PATH,
        "S2"
    )

    if s2_file is None:

        raise FileNotFoundError(
            "\nS2.pkl was not found after extraction.\n"
            "Please check the WESAD ZIP structure."
        )


    dataset_root = os.path.dirname(
        os.path.dirname(s2_file)
    )

    print("\nS2.pkl found:")
    print(s2_file)

    print("\nDataset root:")
    print(dataset_root)

    return dataset_root


# ============================================================
# 4. SIGNAL RESAMPLING
# ============================================================

def resample_signal(
    x,
    original_fs,
    target_fs
):
    """Resample a 1-D signal using scipy.signal.resample."""

    x = np.asarray(
        x,
        dtype=np.float32
    ).squeeze()

    if x.ndim != 1:

        raise ValueError(
            f"Expected 1-D signal, got shape {x.shape}"
        )

    if len(x) == 0:

        return x

    if original_fs == target_fs:

        return x.astype(
            np.float32
        )

    new_length = int(
        round(
            len(x)
            * target_fs
            / original_fs
        )
    )

    new_length = max(
        new_length,
        1
    )

    return signal.resample(
        x,
        new_length
    ).astype(
        np.float32
    )


# ============================================================
# 5. ALIGN 700 Hz LABELS TO 32 Hz
# ============================================================

def align_labels_by_time(
    labels,
    label_fs,
    target_fs,
    target_length
):
    """Convert native 700 Hz labels to target sampling rate."""

    labels = np.asarray(
        labels
    ).squeeze()

    if labels.ndim != 1:

        raise ValueError(
            f"Labels must be 1-D, got {labels.shape}"
        )

    if len(labels) == 0:

        raise ValueError(
            "Labels array is empty."
        )

    target_times = (
        np.arange(
            target_length,
            dtype=np.float64
        )
        / float(target_fs)
    )

    indices = np.floor(
        target_times
        * label_fs
    ).astype(
        np.int64
    )

    indices = np.clip(
        indices,
        0,
        len(labels) - 1
    )

    return labels[
        indices
    ]


# ============================================================
# 6. HRV FEATURES
# ============================================================

def extract_hrv_features(
    bvp_window,
    fs
):
    """
    Extract:
    1. Mean RR
    2. SDNN
    3. RMSSD
    """

    try:

        bvp_window = np.asarray(
            bvp_window,
            dtype=np.float32
        ).squeeze()

        if len(bvp_window) < 3:

            return np.zeros(
                3,
                dtype=np.float32
            )

        if not np.all(
            np.isfinite(bvp_window)
        ):

            finite_mask = np.isfinite(
                bvp_window
            )

            if finite_mask.sum() < 3:

                return np.zeros(
                    3,
                    dtype=np.float32
                )

            bvp_window = np.interp(
                np.arange(
                    len(bvp_window)
                ),
                np.flatnonzero(
                    finite_mask
                ),
                bvp_window[
                    finite_mask
                ]
            )

        min_distance = max(
            1,
            int(
                0.4 * fs
            )
        )

        signal_std = float(
            np.std(
                bvp_window
            )
        )

        if signal_std > 0:

            prominence = (
                0.10
                * signal_std
            )

            peaks, _ = find_peaks(
                bvp_window,
                distance=min_distance,
                prominence=prominence
            )

        else:

            peaks, _ = find_peaks(
                bvp_window,
                distance=min_distance
            )

        if len(peaks) < 3:

            return np.zeros(
                3,
                dtype=np.float32
            )

        rr = (
            np.diff(peaks)
            .astype(np.float64)
            / float(fs)
        )

        rr = rr[
            (rr >= 0.3)
            &
            (rr <= 2.0)
        ]

        if len(rr) < 2:

            return np.zeros(
                3,
                dtype=np.float32
            )

        mean_rr = np.mean(
            rr
        )

        sdnn = np.std(
            rr
        )

        diff_rr = np.diff(
            rr
        )

        if len(diff_rr) > 0:

            rmssd = np.sqrt(
                np.mean(
                    diff_rr ** 2
                )
            )

        else:

            rmssd = 0.0

        features = np.array(
            [
                mean_rr,
                sdnn,
                rmssd
            ],
            dtype=np.float32
        )

        if not np.all(
            np.isfinite(features)
        ):

            return np.zeros(
                3,
                dtype=np.float32
            )

        return features

    except Exception:

        return np.zeros(
            3,
            dtype=np.float32
        )


# ============================================================
# 7. CREATE 60-SECOND WINDOWS
# ============================================================

def create_windows_with_features(
    eda,
    bvp,
    labels,
    fs,
    window_size_s=60,
    step_size_s=10
):
    """Create sequence, HRV and label arrays."""

    eda = np.asarray(
        eda,
        dtype=np.float32
    ).squeeze()

    bvp = np.asarray(
        bvp,
        dtype=np.float32
    ).squeeze()

    labels = np.asarray(
        labels
    ).squeeze()

    common_length = min(
        len(eda),
        len(bvp),
        len(labels)
    )

    eda = eda[
        :common_length
    ]

    bvp = bvp[
        :common_length
    ]

    labels = labels[
        :common_length
    ]

    window_samples = int(
        window_size_s
        * fs
    )

    step_samples = int(
        step_size_s
        * fs
    )

    if common_length < window_samples:

        return (
            np.empty(
                (
                    0,
                    window_samples,
                    2
                ),
                dtype=np.float32
            ),

            np.empty(
                (
                    0,
                    3
                ),
                dtype=np.float32
            ),

            np.empty(
                (
                    0,
                ),
                dtype=np.int64
            )
        )

    sequences = []
    hrv_features = []
    window_labels = []

    for start in range(
        0,
        common_length
        - window_samples
        + 1,
        step_samples
    ):

        end = (
            start
            + window_samples
        )

        eda_win = eda[
            start:end
        ]

        bvp_win = bvp[
            start:end
        ]

        label_win = labels[
            start:end
        ]

        valid_labels = label_win[
            np.isin(
                label_win,
                [1, 2, 3]
            )
        ]

        if len(valid_labels) == 0:

            continue

        values, counts = np.unique(
            valid_labels,
            return_counts=True
        )

        majority_label = int(
            values[
                np.argmax(counts)
            ]
        )

        sequence = np.column_stack(
            [
                eda_win,
                bvp_win
            ]
        ).astype(
            np.float32
        )

        hrv = extract_hrv_features(
            bvp_win,
            fs
        )

        class_label = (
            majority_label - 1
        )

        sequences.append(
            sequence
        )

        hrv_features.append(
            hrv
        )

        window_labels.append(
            class_label
        )

    if len(sequences) == 0:

        return (
            np.empty(
                (
                    0,
                    window_samples,
                    2
                ),
                dtype=np.float32
            ),

            np.empty(
                (
                    0,
                    3
                ),
                dtype=np.float32
            ),

            np.empty(
                (
                    0,
                ),
                dtype=np.int64
            )
        )

    return (
        np.stack(
            sequences
        ).astype(
            np.float32
        ),

        np.stack(
            hrv_features
        ).astype(
            np.float32
        ),

        np.asarray(
            window_labels,
            dtype=np.int64
        )
    )


# ============================================================
# 8. CNN + BATCH NORMALIZATION + DROPOUT
#    + TRANSFORMER + HRV
# ============================================================

class StressClassifier(nn.Module):
    """
    Architecture:

    Input EDA + BVP
          |
       Conv1D
          |
     BatchNorm1D
          |
        ReLU
          |
      MaxPool1D
          |
       Dropout
          |
       Conv1D
          |
     BatchNorm1D
          |
        ReLU
          |
      MaxPool1D
          |
       Dropout
          |
       Conv1D
          |
     BatchNorm1D
          |
        ReLU
          |
      MaxPool1D
          |
       Dropout
          |
    Transformer Encoder
          |
    Global Average Pooling
          |
    Concatenate 3 HRV features
          |
       Dense 64
          |
        ReLU
          |
      Dropout
          |
       Output (3 classes)

    IMPORTANT:
    - WESAD input remains (Batch, 1920, 2).
    - 1920 = 60 seconds × 32 Hz.
    - After three MaxPool1D(2):
      1920 -> 960 -> 480 -> 240.
    - Therefore Transformer receives 240 time-steps.
    """

    def __init__(
        self,
        seq_channels=2,
        hrv_dim=3,
        n_classes=3
    ):

        super().__init__()


        # ----------------------------------------------------
        # CNN feature extraction
        #
        # Conv1D -> BatchNorm -> ReLU
        # -> MaxPool -> Dropout
        # ----------------------------------------------------

        self.cnn = nn.Sequential(

            # Block 1
            nn.Conv1d(
                in_channels=seq_channels,
                out_channels=32,
                kernel_size=7,
                padding=3
            ),

            nn.BatchNorm1d(
                32
            ),

            nn.ReLU(),

            nn.MaxPool1d(
                kernel_size=2
            ),

            nn.Dropout(
                p=0.2
            ),


            # Block 2
            nn.Conv1d(
                in_channels=32,
                out_channels=64,
                kernel_size=5,
                padding=2
            ),

            nn.BatchNorm1d(
                64
            ),

            nn.ReLU(),

            nn.MaxPool1d(
                kernel_size=2
            ),

            nn.Dropout(
                p=0.2
            ),


            # Block 3
            nn.Conv1d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm1d(
                128
            ),

            nn.ReLU(),

            nn.MaxPool1d(
                kernel_size=2
            ),

            nn.Dropout(
                p=0.2
            )
        )


        # ----------------------------------------------------
        # Transformer Encoder
        # ----------------------------------------------------

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=128,
                nhead=TRANSFORMER_HEADS,
                dim_feedforward=256,
                dropout=TRANSFORMER_DROPOUT,
                activation="gelu",
                batch_first=True,
                norm_first=True
            )
        )

        self.transformer = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=TRANSFORMER_LAYERS
            )
        )


        # ----------------------------------------------------
        # Global pooling AFTER Transformer
        # ----------------------------------------------------

        self.global_pool = (
            nn.AdaptiveAvgPool1d(1)
        )


        # ----------------------------------------------------
        # HRV fusion + classifier
        #
        # 128 features + 3 HRV = 131
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.Linear(
                128 + hrv_dim,
                64
            ),

            nn.ReLU(),

            nn.Dropout(
                0.3
            ),

            nn.Linear(
                64,
                n_classes
            )
        )


    def forward(
        self,
        x_seq,
        x_hrv
    ):

        # Input:
        # [batch, time, channels]
        #
        # Conv1D expects:
        # [batch, channels, time]

        x = x_seq.permute(
            0,
            2,
            1
        )


        # CNN
        x = self.cnn(
            x
        )


        # Transformer expects:
        # [batch, time, features]

        x = x.permute(
            0,
            2,
            1
        )


        # Transformer

        x = self.transformer(
            x
        )


        # Back to:
        # [batch, features, time]

        x = x.permute(
            0,
            2,
            1
        )


        # Global average pooling
        # [batch, 128, time]
        # ->
        # [batch, 128, 1]

        x = self.global_pool(
            x
        )


        # [batch, 128]

        x = x.squeeze(
            -1
        )


        # HRV fusion
        # [batch, 128]
        # +
        # [batch, 3]
        # =
        # [batch, 131]

        x = torch.cat(
            [
                x,
                x_hrv
            ],
            dim=1
        )


        # Final classification

        return self.classifier(
            x
        )


# ============================================================
# 9. WESAD PIPELINE
# ============================================================

class WESADPipeline:

    def __init__(
        self,
        dataset_path
    ):

        self.dataset_path = (
            dataset_path
        )

        self.fs = (
            TARGET_SAMPLING_RATE
        )


    def load_subject_raw(
        self,
        subject
    ):
        """Load wrist EDA, wrist BVP and labels."""

        file_path = find_subject_file(
            self.dataset_path,
            subject
        )

        if file_path is None:

            raise FileNotFoundError(
                f"\n{subject}.pkl not found under:\n"
                f"{self.dataset_path}"
            )


        print(
            f"\nLoading {subject}"
        )

        print(
            f"File: {file_path}"
        )


        with open(
            file_path,
            "rb"
        ) as f:

            data = pickle.load(
                f,
                encoding="latin1"
            )


        if "signal" not in data:

            raise KeyError(
                f"{subject}: 'signal' key not found."
            )


        if "wrist" not in data["signal"]:

            raise KeyError(
                f"{subject}: wrist signal not found."
            )


        wrist = data[
            "signal"
        ][
            "wrist"
        ]


        if "EDA" not in wrist:

            raise KeyError(
                f"{subject}: wrist EDA not found."
            )


        if "BVP" not in wrist:

            raise KeyError(
                f"{subject}: wrist BVP not found."
            )


        if "label" not in data:

            raise KeyError(
                f"{subject}: label not found."
            )


        eda_raw = np.asarray(
            wrist["EDA"],
            dtype=np.float32
        ).squeeze()


        bvp_raw = np.asarray(
            wrist["BVP"],
            dtype=np.float32
        ).squeeze()


        labels = np.asarray(
            data["label"]
        ).squeeze()


        # EDA: 4 Hz -> 32 Hz

        eda = resample_signal(
            eda_raw,
            4,
            self.fs
        )


        # BVP: 64 Hz -> 32 Hz

        bvp = resample_signal(
            bvp_raw,
            64,
            self.fs
        )


        common_length = min(
            len(eda),
            len(bvp)
        )


        eda = eda[
            :common_length
        ]

        bvp = bvp[
            :common_length
        ]


        # Labels: 700 Hz -> 32 Hz

        labels = align_labels_by_time(
            labels,
            LABEL_NATIVE_RATE,
            self.fs,
            common_length
        )


        return (
            eda,
            bvp,
            labels
        )


    def build_all_subject_windows(
        self
    ):
        """Build windows for all WESAD subjects."""

        all_sequences = []
        all_hrv = []
        all_labels = []
        all_groups = []


        for subject in SUBJECTS:

            print(
                "\n"
                + "=" * 70
            )

            print(
                f"PROCESSING {subject}"
            )

            print(
                "=" * 70
            )


            eda, bvp, labels = (
                self.load_subject_raw(
                    subject
                )
            )


            seq, hrv, y = (
                create_windows_with_features(
                    eda,
                    bvp,
                    labels,
                    self.fs,
                    WINDOW_SIZE,
                    STEP_SIZE
                )
            )


            if len(y) == 0:

                print(
                    f"WARNING: No valid windows found "
                    f"for {subject}."
                )

                continue


            all_sequences.append(
                seq
            )

            all_hrv.append(
                hrv
            )

            all_labels.append(
                y
            )


            all_groups.extend(
                [subject] * len(y)
            )


            print(
                f"Windows: {len(y)}"
            )

            print(
                f"Sequence shape: {seq.shape}"
            )

            print(
                f"HRV shape: {hrv.shape}"
            )


        if len(all_sequences) == 0:

            raise RuntimeError(
                "No windows were created "
                "from any subject."
            )


        X_seq = np.concatenate(
            all_sequences,
            axis=0
        )


        X_hrv = np.concatenate(
            all_hrv,
            axis=0
        )


        y = np.concatenate(
            all_labels,
            axis=0
        )


        groups = np.asarray(
            all_groups
        )


        return (
            X_seq,
            X_hrv,
            y,
            groups
        )


# ============================================================
# 10. TRAIN ONE LOSO FOLD
# ============================================================

def train_one_fold(
    X_train_seq,
    X_test_seq,
    X_train_hrv,
    X_test_hrv,
    y_train,
    y_test
):
    """Train and evaluate one LOSO fold."""

    # --------------------------------------------------------
    # Sequence scaler: training data only
    # --------------------------------------------------------

    seq_scaler = StandardScaler()


    X_train_seq = (
        seq_scaler.fit_transform(
            X_train_seq.reshape(
                -1,
                X_train_seq.shape[-1]
            )
        )
        .reshape(
            X_train_seq.shape
        )
    )


    X_test_seq = (
        seq_scaler.transform(
            X_test_seq.reshape(
                -1,
                X_test_seq.shape[-1]
            )
        )
        .reshape(
            X_test_seq.shape
        )
    )


    # --------------------------------------------------------
    # HRV scaler: training data only
    # --------------------------------------------------------

    hrv_scaler = StandardScaler()


    X_train_hrv = (
        hrv_scaler.fit_transform(
            X_train_hrv
        )
    )


    X_test_hrv = (
        hrv_scaler.transform(
            X_test_hrv
        )
    )


    # --------------------------------------------------------
    # Convert to tensors
    # --------------------------------------------------------

    X_train_seq = torch.tensor(
        X_train_seq,
        dtype=torch.float32
    )


    X_test_seq = torch.tensor(
        X_test_seq,
        dtype=torch.float32
    )


    X_train_hrv = torch.tensor(
        X_train_hrv,
        dtype=torch.float32
    )


    X_test_hrv = torch.tensor(
        X_test_hrv,
        dtype=torch.float32
    )


    y_train_tensor = torch.tensor(
        y_train,
        dtype=torch.long
    )


    y_test_tensor = torch.tensor(
        y_test,
        dtype=torch.long
    )


    # --------------------------------------------------------
    # DataLoader
    # --------------------------------------------------------

    train_dataset = TensorDataset(
        X_train_seq,
        X_train_hrv,
        y_train_tensor
    )


    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )


    # --------------------------------------------------------
    # Class weights
    # --------------------------------------------------------

    classes = np.unique(
        y_train
    )


    class_weights = np.ones(
        3,
        dtype=np.float32
    )


    calculated_weights = (
        compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=y_train
        )
    )


    class_weights[classes] = (
        calculated_weights
    )


    class_weights = torch.tensor(
        class_weights,
        dtype=torch.float32,
        device=DEVICE
    )


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = StressClassifier(
        seq_channels=2,
        hrv_dim=3,
        n_classes=3
    ).to(
        DEVICE
    )


    # --------------------------------------------------------
    # Loss + Optimizer
    # --------------------------------------------------------

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )


    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for epoch in range(
        1,
        EPOCHS + 1
    ):

        model.train()


        running_loss = 0.0
        num_batches = 0


        for (
            batch_seq,
            batch_hrv,
            batch_y
        ) in train_loader:

            batch_seq = batch_seq.to(
                DEVICE
            )

            batch_hrv = batch_hrv.to(
                DEVICE
            )

            batch_y = batch_y.to(
                DEVICE
            )


            optimizer.zero_grad()


            outputs = model(
                batch_seq,
                batch_hrv
            )


            loss = criterion(
                outputs,
                batch_y
            )


            loss.backward()


            optimizer.step()


            running_loss += (
                loss.item()
            )

            num_batches += 1


        avg_loss = (
            running_loss
            / max(
                num_batches,
                1
            )
        )


        print(
            f"Epoch {epoch:02d}/{EPOCHS} "
            f"| Loss: {avg_loss:.4f}"
        )


    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    model.eval()


    with torch.no_grad():

        outputs = model(
            X_test_seq.to(
                DEVICE
            ),
            X_test_hrv.to(
                DEVICE
            )
        )


        predictions = torch.argmax(
            outputs,
            dim=1
        ).cpu().numpy()


    accuracy = accuracy_score(
        y_test,
        predictions
    )


    return (
        accuracy,
        y_test,
        predictions
    )


# ============================================================
# 11. LOSO VALIDATION
# ============================================================

def run_loso(
    X_seq,
    X_hrv,
    y,
    groups
):
    """Leave-One-Subject-Out validation."""

    logo = LeaveOneGroupOut()


    fold_accuracies = []
    fold_reports = []


    splits = logo.split(
        X_seq,
        y,
        groups
    )


    for fold, (
        train_idx,
        test_idx
    ) in enumerate(
        splits,
        start=1
    ):

        test_subject = (
            groups[test_idx][0]
        )


        print(
            "\n"
            + "=" * 70
        )


        print(
            f"LOSO FOLD {fold}"
        )


        print(
            f"Test Subject: {test_subject}"
        )


        print(
            "=" * 70
        )


        X_train_seq = X_seq[
            train_idx
        ]

        X_test_seq = X_seq[
            test_idx
        ]


        X_train_hrv = X_hrv[
            train_idx
        ]

        X_test_hrv = X_hrv[
            test_idx
        ]


        y_train = y[
            train_idx
        ]

        y_test = y[
            test_idx
        ]


        print(
            f"Train windows: {len(y_train)}"
        )


        print(
            f"Test windows: {len(y_test)}"
        )


        accuracy, y_true, predictions = (
            train_one_fold(
                X_train_seq,
                X_test_seq,
                X_train_hrv,
                X_test_hrv,
                y_train,
                y_test
            )
        )


        fold_accuracies.append(
            accuracy
        )


        print(
            f"\nFold Accuracy: "
            f"{accuracy * 100:.2f}%"
        )


        report = classification_report(
            y_true,
            predictions,
            labels=[
                0,
                1,
                2
            ],
            target_names=[
                "Baseline",
                "Stress",
                "Amusement"
            ],
            zero_division=0
        )


        fold_reports.append(
            report
        )


        print(
            "\nClassification Report:"
        )

        print(
            report
        )


    # ========================================================
    # FINAL RESULTS
    # ========================================================

    fold_accuracies = np.asarray(
        fold_accuracies,
        dtype=np.float64
    )


    mean_accuracy = np.mean(
        fold_accuracies
    )


    std_accuracy = np.std(
        fold_accuracies
    )


    print(
        "\n"
        + "=" * 70
    )


    print(
        "FINAL LOSO RESULTS"
    )


    print(
        "=" * 70
    )


    for i, acc in enumerate(
        fold_accuracies,
        start=1
    ):

        print(
            f"Fold {i:02d}: "
            f"{acc * 100:.2f}%"
        )


    print(
        "-" * 70
    )


    print(
        f"Mean Accuracy: "
        f"{mean_accuracy * 100:.2f}%"
    )


    print(
        f"Std Accuracy: "
        f"{std_accuracy * 100:.2f}%"
    )


    print(
        "=" * 70
    )


    return (
        fold_accuracies,
        fold_reports
    )


# ============================================================
# 12. MAIN
# ============================================================

def main():

    print(
        "\n"
        + "=" * 70
    )


    print(
        "WESAD STRESS DETECTION PIPELINE"
    )


    print(
        "=" * 70
    )


    print(
        f"Device: {DEVICE}"
    )


    print(
        f"Target Sampling Rate: "
        f"{TARGET_SAMPLING_RATE} Hz"
    )


    print(
        f"Window: "
        f"{WINDOW_SIZE} seconds"
    )


    print(
        f"Step: "
        f"{STEP_SIZE} seconds"
    )


    print(
        "Input: EDA + BVP"
    )


    print(
        "HRV Features: Mean RR, SDNN, RMSSD"
    )


    print(
        "Model: 3-Layer 1D CNN + BatchNorm "
        "+ Dropout + Transformer + HRV"
    )


    print(
        "Classes: Baseline, Stress, Amusement"
    )


    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    dataset_path = (
        download_wesad_dataset()
    )


    pipeline = WESADPipeline(
        dataset_path
    )


    X_seq, X_hrv, y, groups = (
        pipeline.build_all_subject_windows()
    )


    # ========================================================
    # DATASET SUMMARY
    # ========================================================

    print(
        "\n"
        + "=" * 70
    )


    print(
        "DATASET SUMMARY"
    )


    print(
        "=" * 70
    )


    print(
        f"X_seq shape : {X_seq.shape}"
    )


    print(
        f"X_hrv shape : {X_hrv.shape}"
    )


    print(
        f"y shape     : {y.shape}"
    )


    print(
        f"groups shape: {groups.shape}"
    )


    print(
        "\nClass distribution:"
    )


    class_names = [
        "Baseline",
        "Stress",
        "Amusement"
    ]


    for class_id, class_name in enumerate(
        class_names
    ):

        count = int(
            np.sum(
                y == class_id
            )
        )


        print(
            f"{class_name}: "
            f"{count}"
        )


    print(
        "\nSubjects found:"
    )


    print(
        np.unique(
            groups
        )
    )


    # ========================================================
    # LOSO
    # ========================================================

    results = run_loso(
        X_seq,
        X_hrv,
        y,
        groups
    )


    return results


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    fold_accuracies, fold_reports = main()

MOUNTING GOOGLE DRIVE
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Google Drive mounted.
WESAD storage location:
/content/drive/MyDrive/WESAD

WESAD STRESS DETECTION PIPELINE
Device: cuda
Target Sampling Rate: 32 Hz
Window: 60 seconds
Step: 10 seconds
Input: EDA + BVP
HRV Features: Mean RR, SDNN, RMSSD
Model: 3-Layer 1D CNN + BatchNorm + Dropout + Transformer + HRV
Classes: Baseline, Stress, Amusement
WESAD DATASET SETUP

WESAD ZIP already exists in Google Drive.
/content/drive/MyDrive/WESAD/WESAD.zip

Extracting WESAD ZIP to Google Drive...
Extraction completed.

S2.pkl found:
/content/drive/MyDrive/WESAD/WESAD/S2/S2.pkl

Dataset root:
/content/drive/MyDrive/WESAD/WESAD

PROCESSING S2

Loading S2
File: /content/drive/MyDrive/WESAD/WESAD/S2/S2.pkl
Windows: 230
Sequence shape: (230, 1920, 2)
HRV shape: (230, 3)

PROCESSING S3

Loading S3
File: /content/drive/MyDrive/WESAD/WESAD/S3/S3.pkl
Windows: 233
Se

/tmp/ipykernel_2906/2608965975.py:962: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(


Epoch 01/15 | Loss: 0.9046
Epoch 02/15 | Loss: 0.6830
Epoch 03/15 | Loss: 0.5297
Epoch 04/15 | Loss: 0.4670
Epoch 05/15 | Loss: 0.4249
Epoch 06/15 | Loss: 0.3909
Epoch 07/15 | Loss: 0.3539
Epoch 08/15 | Loss: 0.3602
Epoch 09/15 | Loss: 0.4068
Epoch 10/15 | Loss: 0.3161
Epoch 11/15 | Loss: 0.2937
Epoch 12/15 | Loss: 0.3117
Epoch 13/15 | Loss: 0.3123
Epoch 14/15 | Loss: 0.2624
Epoch 15/15 | Loss: 0.2321

Fold Accuracy: 36.33%

Classification Report:
              precision    recall  f1-score   support

    Baseline       0.20      0.09      0.12       124
      Stress       0.41      1.00      0.58        78
   Amusement       0.00      0.00      0.00        43

    accuracy                           0.36       245
   macro avg       0.21      0.36      0.24       245
weighted avg       0.23      0.36      0.25       245


LOSO FOLD 2
Test Subject: S11
Train windows: 3336
Test windows: 241


/tmp/ipykernel_2906/2608965975.py:962: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(


Epoch 01/15 | Loss: 0.9198
Epoch 02/15 | Loss: 0.7184
Epoch 03/15 | Loss: 0.6182
Epoch 04/15 | Loss: 0.5342
Epoch 05/15 | Loss: 0.4732
Epoch 06/15 | Loss: 0.4461
Epoch 07/15 | Loss: 0.4170
Epoch 08/15 | Loss: 0.3871
Epoch 09/15 | Loss: 0.3756
Epoch 10/15 | Loss: 0.3667
Epoch 11/15 | Loss: 0.3066
Epoch 12/15 | Loss: 0.3096
Epoch 13/15 | Loss: 0.2987
Epoch 14/15 | Loss: 0.2820
Epoch 15/15 | Loss: 0.2767

Fold Accuracy: 70.54%

Classification Report:
              precision    recall  f1-score   support

    Baseline       0.98      0.45      0.62       124
      Stress       0.97      1.00      0.99        74
   Amusement       0.37      0.93      0.53        43

    accuracy                           0.71       241
   macro avg       0.78      0.79      0.71       241
weighted avg       0.87      0.71      0.72       241


LOSO FOLD 3
Test Subject: S13
Train windows: 3337
Test windows: 240


...

Epoch 01/15 | Loss: 0.8658
Epoch 02/15 | Loss: 0.6520
Epoch 03/15 | Loss: 0.5618
Epoch 04/15 | Loss: 0.4722
Epoch 05/15 | Loss: 0.4526
Epoch 06/15 | Loss: 0.3872
Epoch 07/15 | Loss: 0.4184
Epoch 08/15 | Loss: 0.3671
Epoch 09/15 | Loss: 0.3440
Epoch 10/15 | Loss: 0.3088
Epoch 11/15 | Loss: 0.2889
Epoch 12/15 | Loss: 0.3115
Epoch 13/15 | Loss: 0.2810
Epoch 14/15 | Loss: 0.2682
Epoch 15/15 | Loss: 0.2299

Fold Accuracy: 44.73%

Classification Report:
              precision    recall  f1-score   support

    Baseline       0.88      0.30      0.45       124
      Stress       0.38      0.99      0.55        70
   Amusement       0.00      0.00      0.00        43

    accuracy                           0.45       237
   macro avg       0.42      0.43      0.33       237
weighted avg       0.57      0.45      0.40       237


LOSO FOLD 14
Test Subject: S8
Train windows: 3338
Test windows: 239


/tmp/ipykernel_2906/2608965975.py:962: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(


Epoch 01/15 | Loss: 0.9314
Epoch 02/15 | Loss: 0.7411
Epoch 03/15 | Loss: 0.5801
Epoch 04/15 | Loss: 0.5029
Epoch 05/15 | Loss: 0.4470
Epoch 06/15 | Loss: 0.4161
Epoch 07/15 | Loss: 0.3768
Epoch 08/15 | Loss: 0.3545
Epoch 09/15 | Loss: 0.3197
Epoch 10/15 | Loss: 0.3419
Epoch 11/15 | Loss: 0.3011
Epoch 12/15 | Loss: 0.2996
Epoch 13/15 | Loss: 0.2923
Epoch 14/15 | Loss: 0.3040
Epoch 15/15 | Loss: 0.2904

Fold Accuracy: 81.59%

Classification Report:
              precision    recall  f1-score   support

    Baseline       0.74      0.99      0.85       123
      Stress       1.00      1.00      1.00        73
   Amusement       0.00      0.00      0.00        43

    accuracy                           0.82       239
   macro avg       0.58      0.66      0.62       239
weighted avg       0.69      0.82      0.74       239


LOSO FOLD 15
Test Subject: S9
Train windows: 3339
Test windows: 238


/tmp/ipykernel_2906/2608965975.py:962: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(


Epoch 01/15 | Loss: 0.8920
Epoch 02/15 | Loss: 0.6530
Epoch 03/15 | Loss: 0.5418
Epoch 04/15 | Loss: 0.5075
Epoch 05/15 | Loss: 0.4339
Epoch 06/15 | Loss: 0.4062
Epoch 07/15 | Loss: 0.3849
Epoch 08/15 | Loss: 0.3658
Epoch 09/15 | Loss: 0.3571
Epoch 10/15 | Loss: 0.3471
Epoch 11/15 | Loss: 0.3338
Epoch 12/15 | Loss: 0.3429
Epoch 13/15 | Loss: 0.2904
Epoch 14/15 | Loss: 0.2813
Epoch 15/15 | Loss: 0.2897

Fold Accuracy: 79.41%

Classification Report:
              precision    recall  f1-score   support

    Baseline       0.73      0.98      0.84       124
      Stress       1.00      0.57      0.73        70
   Amusement       0.90      0.61      0.73        44

    accuracy                           0.79       238
   macro avg       0.88      0.72      0.76       238
weighted avg       0.84      0.79      0.78       238


FINAL LOSO RESULTS
Fold 01: 36.33%
Fold 02: 70.54%
Fold 03: 61.67%
Fold 04: 64.98%
Fold 05: 62.24%
Fold 06: 82.92%
Fold 07: 55.92%
Fold 08: 71.30%
Fold 09: 60.52%
Fol